# 🇨🇴 Colombia INS (Instituto Nacional de Salud) Data Access

This notebook demonstrates how to access public health surveillance data from **Colombia's National Health Institute (INS)**, including SIVIGILA (Public Health Surveillance System) data.

## Overview

The Colombian INS manages the SIVIGILA system for notifiable disease surveillance, covering:
- **Vector-borne diseases**: Dengue, Malaria, Chikungunya, Zika, Yellow fever
- **Respiratory diseases**: COVID-19, Influenza, IRA
- **Water and food-borne diseases**: Hepatitis, EDA
- **Sexually transmitted infections**: VIH/SIDA, Syphilis
- **Tuberculosis and Leprosy**
- **Public health events of interest**: Suicide attempts, violence, traffic accidents

**Website:** https://www.ins.gov.co/

**Open Data Portal:** https://www.datos.gov.co

## 📦 Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Add scripts directory to path
sys.path.insert(0, str(Path.cwd().parent.parent / 'scripts'))

from accessors import ColombiaINSAccessor

# Initialize accessor
ins = ColombiaINSAccessor()

print("✅ Colombia INS Accessor initialized successfully!")

## 🗺️ Exploring Colombian Geography

Colombia is divided into 33 departments (departamentos) organized into 5 geographical regions:

In [ ]:
# List all departments
departments = ins.list_departments()
print(f"Total departments: {len(departments)}")
print("\nFirst 10 departments:")
departments.head(10)

In [ ]:
# View departments by region
regions = {
    "Andina": ins.get_departments_by_region('Andina'),
    "Caribe": ins.get_departments_by_region('Caribe'),
    "Pacífica": ins.get_departments_by_region('Pacífica'),
    "Orinoquía": ins.get_departments_by_region('Orinoquía'),
    "Amazonía": ins.get_departments_by_region('Amazonía'),
}

for region, depts in regions.items():
    dept_names = [ins.DEPARTMENTS[d] for d in depts]
    print(f"\n📍 {region} ({len(depts)} departments):")
    print("  " + ", ".join(dept_names[:5]) + ("..." if len(dept_names) > 5 else ""))

## 🦠 Notifiable Diseases (SIVIGILA)

SIVIGILA tracks 33 notifiable diseases organized into event groups:

In [ ]:
# List all diseases
diseases = ins.list_diseases()
print(f"Total notifiable diseases: {len(diseases)}\n")
diseases.head(10)

In [ ]:
# View diseases by event group
event_groups = ins.list_event_groups()

for group in event_groups['event_group'].unique():
    group_diseases = event_groups[event_groups['event_group'] == group]
    print(f"\n📋 {group} ({len(group_diseases)} diseases):")
    for _, row in group_diseases.head(5).iterrows():
        print(f"  - {row['disease_code']}: {row['disease_name']}")
    if len(group_diseases) > 5:
        print(f"  ... and {len(group_diseases) - 5} more")

## 🦟 Dengue Surveillance

Dengue is a priority disease in Colombia with mandatory weekly reporting. Let's access dengue data:

In [ ]:
# Get dengue data for recent years
# Note: Actual data requires integration with datos.gov.co API
dengue_data = ins.get_dengue_data(
    years=[2022, 2023],
    departments=['05', '76', '11']  # Antioquia, Valle del Cauca, Bogotá
)

print(f"Dengue data records: {len(dengue_data)}")
print("\nDengue types tracked:")
print(dengue_data[['disease_code', 'disease_name']].drop_duplicates())

In [ ]:
# Dengue by department
print("\nDengue surveillance by department:")
dengue_by_dept = dengue_data.groupby('department_name').size().reset_index(name='records')
dengue_by_dept

## 🦟 Malaria Surveillance

Malaria is endemic in several Colombian regions, particularly the Pacific coast and Amazon:

In [ ]:
# Get malaria data for Amazon and Pacific regions
amazon_depts = ins.get_departments_by_region('Amazonía')
pacific_depts = ins.get_departments_by_region('Pacífica')

malaria_data = ins.get_malaria_data(
    years=[2022, 2023],
    departments=amazon_depts + pacific_depts
)

print(f"Malaria data records: {len(malaria_data)}")
print("\nMalaria types tracked:")
print(malaria_data[['disease_code', 'disease_name']].drop_duplicates())

## 🦠 Arbovirus Surveillance (Dengue, Chikungunya, Zika, Yellow Fever)

Colombia conducts integrated surveillance for arboviruses due to similar transmission patterns:

In [ ]:
# Get all arbovirus data
arbovirus_data = ins.get_arbovirus_data(
    years=[2023],
    departments=ins.get_departments_by_region('Caribe')
)

print(f"Arbovirus surveillance records: {len(arbovirus_data)}")
print("\nArboviruses tracked:")
arbovirus_summary = arbovirus_data[['disease_code', 'disease_name']].drop_duplicates()
arbovirus_summary

## 📊 Weekly Epidemiological Bulletins

The INS publishes weekly epidemiological bulletins (Boletines Epidemiológicos Semanales):

In [ ]:
# Get bulletins for a specific year
bulletins_2023 = ins.get_weekly_bulletins(year=2023)

print(f"Total bulletins in 2023: {len(bulletins_2023)}")
print("\nFirst 5 weeks:")
bulletins_2023.head()[['year', 'week', 'data_source', 'note']]

In [ ]:
# Get specific week bulletin
week_10 = ins.get_weekly_bulletins(year=2023, week=10)
print("Week 10 bulletin:")
week_10[['year', 'week', 'bulletin_url', 'diseases_covered']]

## 📈 Comparing Departments

Compare disease occurrence across departments over time:

In [ ]:
# Compare dengue across major cities
major_cities = ['05', '76', '11', '08', '66']  # Antioquia, Valle, Bogotá, Atlántico, Risaralda

comparison = ins.compare_departments(
    disease='100',  # Dengue
    years=[2022, 2023],
    departments=major_cities
)

print("Dengue comparison structure (sample):")
print(comparison.head() if not comparison.empty else "No data available")

## 📋 Summary by Department

Get summary statistics for a specific year:

In [ ]:
# Get summary for vector-borne diseases
summary = ins.get_summary_by_department(
    year=2023,
    disease_group='ENFERMEDADES TRANSMITIDAS POR VECTORES'
)

print(f"Summary records: {len(summary)}")
print("\nFirst 10 departments:")
summary.head(10) if not summary.empty else "No data available"

## 🔍 Data Access Notes

### Current Implementation

This accessor provides:
- ✅ Complete metadata (departments, diseases, event groups)
- ✅ Structured data access methods
- ✅ Regional and temporal filtering
- ✅ Ready-to-use data structures

### Future Enhancements

For full data access, integrate with:
1. **datos.gov.co API** - Colombia's open data portal
2. **INS SIVIGILA API** - Direct access to surveillance system
3. **PDF parsing** - Extract data from weekly bulletins

### Data Availability

| Data Type | Frequency | Access Method |
|-----------|-----------|---------------|
| Weekly bulletins | Weekly | PDF downloads |
| Case data | Weekly | datos.gov.co API |
| Annual summaries | Yearly | INS reports |
| Geographic data | Static | Administrative divisions |


## 💾 Export Data

Save processed data for further analysis:

In [ ]:
# Create output directory
output_dir = Path('./output')
output_dir.mkdir(exist_ok=True)

# Export departments
departments.to_csv(output_dir / 'colombia_departments.csv', index=False)
print("✅ Exported: colombia_departments.csv")

# Export diseases
diseases.to_csv(output_dir / 'colombia_diseases.csv', index=False)
print("✅ Exported: colombia_diseases.csv")

# Export event groups
event_groups.to_csv(output_dir / 'colombia_event_groups.csv', index=False)
print("✅ Exported: colombia_event_groups.csv")

print(f"\n📁 All files saved to: {output_dir.absolute()}")

## 📚 References

- **INS Colombia**: https://www.ins.gov.co/
- **SIVIGILA**: https://www.sivigila.gov.co
- **Datos Abiertos Colombia**: https://www.datos.gov.co
- **Event Groups Classification**: Resolución 2113 de 2021 (Colombia)

---

**Last Updated:** March 2026  
**Data Source:** Instituto Nacional de Salud (INS) - Colombia